# DroneFuse - Step 3: fusion with missing sensors and unknown-drone rejection (GPU)

Trains and evaluates, on the Step 2 features, for all 9 protocols and 3 seeds:
- **single-sensor** models (camera, audio, Avia, Mid-360, radar) and **late fusion** of them;
- **feature concatenation**, an **LMMUAVC-style** missing-slot model, and **DroneFuse** (attention that skips
  missing sensors, trained by removing sensors in the co-missing patterns MMAUD really shows), plus two
  DroneFuse variants (i.i.d. dropping; mixed dropping);
- unknown-type rejection with max-softmax, energy and **availability-normalised energy**;
- test-time sensor removal: every subset of sensor groups, and extra i.i.d. / pattern removal.

Everything that is chosen (presence threshold, best epoch, rejection thresholds) is chosen on **validation**;
validation never contains an unknown type. The test splits are only scored.

**Kaggle settings:** Accelerator = **GPU T4** (one GPU is enough). Internet = Off is fine (nothing is downloaded).
**Inputs to add:** `dronefuse-code` (latest version) and `dronefuse-step2-features`.
To continue an interrupted run, also add the output of that earlier Step 3 run: finished runs are skipped.
**Safe for "Save & Run All":** one step per protocol; an error is recorded and the next protocol runs;
`step3_report.txt` is always written. **Expected run time:** under 1 GPU-hour.

In [ ]:
import os
import sys
from pathlib import Path

# The DroneFuse code is attached as a Kaggle dataset. Find the `dronefuse`
# package wherever Kaggle placed it, and make it importable.
def find_file(name, root='/kaggle/input'):
    # os.walk with followlinks=True also enters inputs that Kaggle mounts as links.
    for folder, _, files in os.walk(root, followlinks=True):
        if name in files:
            return Path(folder) / name
    return None


package = find_file('rosbag_index.py')
if package is None:
    attached = sorted(str(p) for p in Path('/kaggle/input').glob('*/*'))[:20]
    raise RuntimeError('The dronefuse-code dataset is not attached to this notebook. In the editor: '
                       'Add Input -> Your Datasets -> dronefuse-code, then Save & Run All again. '
                       f'Attached now: {attached or "nothing"}')
if (package.parent / '__init__.py').exists() and package.parent.name == 'dronefuse':
    code_root = package.parent.parent
else:
    # Kaggle sometimes flattens an uploaded folder, so the files sit directly in
    # the dataset with no `dronefuse/` folder around them. In that case, link the
    # folder that holds rosbag_index.py under the name `dronefuse`.
    flat = package
    code_root = Path('/kaggle/working/code')
    code_root.mkdir(parents=True, exist_ok=True)
    if not (code_root / 'dronefuse').exists():
        os.symlink(flat.parent, code_root / 'dronefuse')

sys.path.insert(0, str(code_root))
import dronefuse
print('DroneFuse code loaded from', Path(dronefuse.__file__).parent)

In [ ]:
# ---- Settings (the only cell you should need to edit) ----
from dronefuse.fusion import TrainSettings
from dronefuse.windows import PROTOCOLS

INPUT = Path('/kaggle/input')
OUTPUT = Path('/kaggle/working/dronefuse_outputs/step3')
SEEDS = [0, 1, 2]
RUN_PROTOCOLS = list(PROTOCOLS)     # closed, v1_to_v2, v2_to_v1, then the six leave-one-type-out protocols
TIME_BUDGET_MINUTES = 11 * 60
SETTINGS = TrainSettings(epochs=40, batch_size=256, lr=1e-3, weight_decay=1e-2)

In [ ]:
import json

from dronefuse import environment
from dronefuse.fusion import load_step2
from dronefuse.runner import StepRunner
from dronefuse.step3 import availability_table, device_name, resume_from
from dronefuse.windows import load_windows

OUTPUT.mkdir(parents=True, exist_ok=True)
runner = StepRunner(OUTPUT, TIME_BUDGET_MINUTES)
summary = {}


def save_summary():
    summary['steps'] = runner.steps
    with open(OUTPUT / 'step3_summary.json', 'w') as f:
        json.dump(summary, f, indent=1, default=str)


summary['environment'] = runner.run('measure the machine', environment.collect)
windows_csv = code_root / 'splits' / 'windows.csv'
windows = load_windows(windows_csv)

# The Step 2 features: the folder that holds the <recording>.deep.npz files.
deep_files = sorted(p for p in INPUT.rglob('*.deep.npz') if 'runs' not in p.parts)
if not deep_files:
    raise RuntimeError('No Step 2 features found. Add Input -> Your Datasets -> dronefuse-step2-features, '
                       'then Save & Run All again.')
STEP2 = deep_files[0].parent
print('Step 2 features:', STEP2, f'({len(deep_files)} recordings)')

# An earlier Step 3 output attached as input: copy its finished runs, so they are not repeated.
for previous in sorted({p.parent.parent for p in INPUT.rglob('runs/*_threshold.json')}):
    resume_from(previous, OUTPUT)

data = runner.run('load the Step 2 features', load_step2, STEP2, windows)
if data is not None:
    print(len(data), 'covered windows; skipped:', data.skipped or 'none')
    summary['windows'] = len(data)
    summary['skipped_recordings'] = data.skipped
    summary['availability_any_point'] = availability_table(data, 1)
    print('sensor group present (fraction of windows, any point):', json.dumps(summary['availability_any_point']))
print('device:', device_name())
save_summary()

In [ ]:
# The experiments: one step per protocol (a few minutes each on a GPU).
from dronefuse.step3 import run_protocol

summary['protocols'] = {}
if data is not None:
    for protocol in RUN_PROTOCOLS:
        summary['protocols'][protocol] = runner.run(f'protocol {protocol}', run_protocol, data, windows, protocol,
                                                    SEEDS, OUTPUT, SETTINGS)
        save_summary()

In [ ]:
# Tables, figures and the report (numbers come only from the runs saved in OUTPUT/runs).
from dronefuse.step3 import format_step3_report, summarize, write_figures, write_tables

results = runner.run('summarise the runs', summarize, data, windows, OUTPUT, RUN_PROTOCOLS, tuple(SEEDS)) \
    if data is not None else None
if results is not None:
    with open(OUTPUT / 'results.json', 'w') as f:
        json.dump(results, f, indent=1)
    summary['tables'] = runner.run('LaTeX tables', write_tables, results, OUTPUT / 'tables')
    summary['figures'] = runner.run('figures', write_figures, results, OUTPUT / 'figures')
    report = format_step3_report(results, data, summary.get('environment'))
    report += '\n\nAVAILABILITY (any point): ' + json.dumps(summary.get('availability_any_point'))
else:
    report = 'NO RESULTS - see the errors above.'
failed = runner.failed()
report += '\n\nSTEPS: ' + ('all succeeded' if not failed else 'not succeeded: ' + ', '.join(failed))
(OUTPUT / 'step3_report.txt').write_text(report)
save_summary()
print(report)

## After the run

1. Send back `dronefuse_outputs/step3/step3_report.txt` (or the printed report above).
2. Open this notebook's saved version -> **Output** -> **New Dataset**, name it **`dronefuse-step3-results`**
   (private). It holds every run (logits included), so new analyses need no retraining.